<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/Phase_2_Patient_Classification/OPEN_WEIGHT_FINAL_SET_UP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
import sys
import psutil
import base64

print('=' * 60)
print('🖥️  HARDWARE & ENVIRONMENT SETUP')
print('=' * 60)

# Check GPU
try:
    gpu = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'], text=True
    ).strip()
    name, total, free = [x.strip() for x in gpu.split(',')]
    print(f'✅ GPU: {name} | VRAM: {int(total)/1024:.1f} GB total')
except Exception as e:
    print(f'❌ GPU check failed. Error: {e}')

# Install CrewAI & Ollama silently
print("\n📦 Installing CrewAI and dependencies (this takes about a minute)...")
# FIX: Added 'litellm' to the pip install list to support the OpenAI-compatible proxy!
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'crewai', 'litellm'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True)

# FIX: Using Base64 encoding to completely hide the URL from Colab's markdown auto-formatter!
url_b64 = b"aHR0cHM6Ly9vbGxhbWEuY29tL2luc3RhbGwuc2g="
ollama_url = base64.b64decode(url_b64).decode('utf-8')
install_cmd = f"curl -fsSL {ollama_url} | sh"

subprocess.run(install_cmd, shell=True, check=True, stdout=subprocess.DEVNULL)
print('✅ Installations complete!')



In [ ]:
import subprocess
import time
import requests
import os

env = os.environ.copy()

env.update({
    'OLLAMA_NUM_GPU': '99',
    'OLLAMA_NUM_PARALLEL': '1',
    'OLLAMA_FLASH_ATTENTION': '1',
    'OLLAMA_MAX_LOADED_MODELS': '3',   # keeps all 3 in VRAM simultaneously
    'OLLAMA_KEEP_ALIVE': '-1',
    'OLLAMA_KV_CACHE_TYPE': 'q8_0',   # NEW: quantised KV cache = faster inference
})

print("🚀 Starting Ollama Server...")
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('ollama.log', 'w'),
    stderr=subprocess.STDOUT,
    env=env
)

for _ in range(30):
    try:
        if requests.get('http://localhost:11434', timeout=1).status_code == 200:
            print('✅ Ollama server is running and ready!')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('❌ Server failed to start. Check ollama.log')

In [ ]:
import subprocess

# A100 80GB ensemble — all 3 fit simultaneously (~57GB total in 4-bit)
# Gemma2:27b ≈ 14GB  | Qwen2.5:14b ≈ 8GB  | Llama3.1:70b ≈ 35GB
MODELS_TO_PULL = [
    "llama3.1:70b",   # Consolidator
    "gemma2:27b",     # Dr. A — swapped back to 27b for better reasoning
    "qwen2.5:14b",    # Dr. B
]

print(f"📥 Pulling ensemble: {MODELS_TO_PULL} (10–15 min first time, cached after)")

for model in MODELS_TO_PULL:
    print(f"\nPulling {model}...")
    result = subprocess.run(['ollama', 'pull', model], capture_output=False)
    if result.returncode == 0:
        print(f"✅ {model} ready!")
    else:
        print(f"❌ Failed to pull {model}")

print("\n📊 VRAM check after loading:")
subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv'])

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "NA"
from crewai import LLM

print("🔗 Connecting CrewAI to Local Ollama Ensemble...")

llm_llama = LLM(
    model="openai/llama3.1:70b",
    base_url="http://localhost:11434/v1",
    api_key="NA",
    timeout=600,
    temperature=0.0,
    num_ctx=8192,
    num_predict=800,
)

llm_gemma = LLM(
    model="openai/gemma2:27b",      # ← back to 27b
    base_url="http://localhost:11434/v1",
    api_key="NA",
    timeout=400,
    temperature=0.0,
    num_ctx=8192,
    num_predict=800,
)

llm_qwen = LLM(
    model="openai/qwen2.5:14b",
    base_url="http://localhost:11434/v1",
    api_key="NA",
    timeout=300,
    temperature=0.0,
    num_ctx=8192,
    num_predict=800,
)

print("✅ Llama 3.1 70B (consolidator) | Gemma 2 27B (Dr. A) | Qwen 2.5 14B (Dr. B)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL A — SIZE CHECK + COMPRESSION AUDIT  (T5 · N=200)
#
# WHAT THIS DOES:
#   1. Loads 200_patients_LLM_final_version_T5.xlsx
#   2. Measures raw record size for every patient (no compression yet)
#   3. Tells you exactly how many patients need compression vs can go raw
#   4. If compression IS needed: uploads MASTER_COMPRESSION_MAP and
#      builds smart_compress_patient() adapted for T5
#   5. Leaves compressed_records_cache in memory for Cell B (classification)
#
# KEY DIFFERENCES FROM OLD PIPELINE:
#   - Index date = reclass_date_5 (T5 = 5 years after first treatment)
#   - Data is already window-sliced in the Excel (no need to re-filter by date)
#   - Column PATIENT used throughout (not Id)
#   - No devices / imaging_studies sheets in this Excel
#   - Ground truth columns (GT_T5_DCSI_TIER etc.) are NOT present — clean file
#
# PREREQUISITES:
#   - 200_patients_LLM_final_version_T5.xlsx uploaded or on Drive
#   - CONSOLIDATED_FRAMEWORK in memory (or will estimate budget without it)
#   - master_compression_map.json (only needed if size check says compression required)
# ══════════════════════════════════════════════════════════════════════════════

import os, re, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files as colab_files

# ─────────────────────────────────────────────────────────────────────────────
# 0. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
INDEX_DATE_LABEL   = "T5_reclassification"   # label used in patient record headers
MAX_OBS_HISTORY    = 5                        # max readings per obs type (compression only)

# Context budget — bottleneck is DeepSeek at 64K tokens
FRAMEWORK_CHARS    = len(CONSOLIDATED_FRAMEWORK) if "CONSOLIDATED_FRAMEWORK" in dir() else 8_000
PROMPT_OVERHEAD    = 3_000
OVERHEAD_TOKENS    = (FRAMEWORK_CHARS + PROMPT_OVERHEAD) // 4
BOTTLENECK_LIMIT   = 64_000
AVAILABLE          = BOTTLENECK_LIMIT - OVERHEAD_TOKENS
SAFE_LIMIT_TOKENS  = int(AVAILABLE * 0.80)
SAFE_LIMIT_CHARS   = SAFE_LIMIT_TOKENS * 4
HARD_MAX_CHARS     = AVAILABLE * 4

print("=" * 65)
print("CELL A — SIZE CHECK + COMPRESSION AUDIT (T5 · N=200)")
print("=" * 65)
print(f"\n  Context budget:")
print(f"    Framework + overhead : ~{OVERHEAD_TOKENS:,} tokens")
print(f"    Safe limit (80%)     : ~{SAFE_LIMIT_TOKENS:,} tokens  /  {SAFE_LIMIT_CHARS:,} chars")
print(f"    Hard ceiling         : ~{AVAILABLE:,} tokens  /  {HARD_MAX_CHARS:,} chars\n")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — UPLOAD AND LOAD EXCEL
# ─────────────────────────────────────────────────────────────────────────────
print("STEP 1 — Upload 200_patients_LLM_final_version_T5.xlsx")
uploaded_xl   = colab_files.upload()
PIPELINE_FILE = list(uploaded_xl.keys())[0]
print(f"  Loaded: {PIPELINE_FILE}")

xl            = pd.ExcelFile(PIPELINE_FILE)
print(f"  Sheets found: {xl.sheet_names}\n")

# Load all sheets — gracefully skip any that don't exist
def load_sheet(name):
    if name in xl.sheet_names:
        return pd.read_excel(PIPELINE_FILE, sheet_name=name)
    return pd.DataFrame()

patient_list  = load_sheet("patient_list")
patients      = load_sheet("patients")
conditions    = load_sheet("conditions")
observations  = load_sheet("observations")
medications   = load_sheet("medications")
encounters    = load_sheet("encounters")
careplans     = load_sheet("careplans")
allergies     = load_sheet("allergies")
procedures    = load_sheet("procedures")
immunizations = load_sheet("immunizations")

ALL_PATIENT_IDS = patient_list["PATIENT"].tolist()
print(f"  Patients loaded : {len(ALL_PATIENT_IDS)}")

# ── Column sanity check ────────────────────────────────────────
print("\n  Column check:")
for sheet_name, df in [
    ("patient_list",  patient_list),
    ("patients",      patients),
    ("conditions",    conditions),
    ("observations",  observations),
    ("medications",   medications),
]:
    pat_col = "PATIENT" if "PATIENT" in df.columns else ("Id" if "Id" in df.columns else "MISSING")
    n_rows  = len(df)
    print(f"    {sheet_name:<15}  {n_rows:>6,} rows   patient col = '{pat_col}'")

# ── Ground truth leak check ────────────────────────────────────
forbidden = ["GT_T5_DCSI", "T5_DCSI_CAT", "GROUND_TRUTH", "DIED_IN_FOLLOWUP",
             "DEATH_DATE", "DEATHDATE"]
leaked = [
    f"{s}.{c}"
    for s in xl.sheet_names
    for c in pd.read_excel(PIPELINE_FILE, sheet_name=s).columns
    if any(f.upper() in str(c).upper() for f in forbidden)
]
if leaked:
    print(f"\n  WARNING — potential ground truth columns detected:")
    for col in leaked:
        print(f"    {col}")
else:
    print(f"\n  No ground truth leakage detected")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — BUILD RAW RECORDS AND MEASURE SIZES
# Data is already window-sliced in the Excel, so no date filtering needed here.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2 — Measuring raw record sizes (no compression)")
print("=" * 65)

PII_COLS = {"SSN", "DRIVERS", "PASSPORT", "ADDRESS", "CITY", "STATE",
            "COUNTY", "ZIP", "LAT", "LON", "HEALTHCARE_EXPENSES",
            "HEALTHCARE_COVERAGE"}

def build_raw_record(patient_id):
    """
    Assemble a full plain-text patient record from all sheets.
    Data is already covariate-window-sliced — no date filtering needed.
    PII columns are stripped; all clinical rows are included as-is.
    """
    record = (
        f"=== PATIENT RECORD: {patient_id} ===\n"
        f"=== INDEX DATE: {INDEX_DATE_LABEL.upper()} "
        f"(5 years after first T2D treatment) ===\n\n"
    )

    # Demographics
    demo = patients[patients.get("PATIENT", patients.get("Id", pd.Series())).eq(patient_id)]
    if "Id" in patients.columns:
        demo = patients[patients["Id"] == patient_id]
    elif "PATIENT" in patients.columns:
        demo = patients[patients["PATIENT"] == patient_id]
    else:
        demo = pd.DataFrame()

    if not demo.empty:
        keep = [c for c in demo.columns if c not in PII_COLS]
        record += f"--- DEMOGRAPHICS ---\n{demo[keep].to_string(index=False)}\n\n"
    else:
        record += "--- DEMOGRAPHICS ---\nNo demographics found.\n\n"

    # Index date metadata from patient_list
    meta = patient_list[patient_list["PATIENT"] == patient_id]
    if not meta.empty:
        record += f"--- INDEX DATE METADATA ---\n{meta.to_string(index=False)}\n\n"

    # Clinical sheets
    clinical = [
        ("CONDITIONS",    conditions,    ["PATIENT", "ENCOUNTER"]),
        ("OBSERVATIONS",  observations,  ["PATIENT", "ENCOUNTER", "TYPE"]),
        ("MEDICATIONS",   medications,   ["PATIENT", "ENCOUNTER", "PAYER"]),
        ("ENCOUNTERS",    encounters,    ["PATIENT"]),
        ("PROCEDURES",    procedures,    ["PATIENT", "ENCOUNTER"]),
        ("CAREPLANS",     careplans,     ["PATIENT", "ENCOUNTER"]),
        ("ALLERGIES",     allergies,     ["PATIENT", "ENCOUNTER"]),
        ("IMMUNIZATIONS", immunizations, ["PATIENT", "ENCOUNTER"]),
    ]

    for section_name, df, drop_cols in clinical:
        if df.empty:
            record += f"--- {section_name} ---\nSheet not present in Excel.\n\n"
            continue
        pat_df = df[df["PATIENT"] == patient_id]
        if not pat_df.empty:
            keep = [c for c in pat_df.columns if c not in drop_cols]
            record += (
                f"--- {section_name} ({len(pat_df)} rows) ---\n"
                f"{pat_df[keep].to_string(index=False)}\n\n"
            )
        else:
            record += f"--- {section_name} ---\nNo records in covariate window.\n\n"

    return record


# Measure all 200 patients
print(f"\n  Building raw records for {len(ALL_PATIENT_IDS)} patients...")
raw_sizes   = []
raw_records = {}

for i, pid in enumerate(ALL_PATIENT_IDS):
    try:
        record = build_raw_record(pid)
        raw_records[pid] = record
        chars  = len(record)
        tokens = chars // 4

        # Sparsity check
        has_conditions   = conditions[conditions["PATIENT"] == pid].shape[0] > 0  if not conditions.empty   else False
        has_observations = observations[observations["PATIENT"] == pid].shape[0] > 0 if not observations.empty else False
        has_medications  = medications[medications["PATIENT"] == pid].shape[0] > 0  if not medications.empty  else False
        n_fields         = sum([has_conditions, has_observations, has_medications])

        raw_sizes.append({
            "patient_id":         pid,
            "raw_chars":          chars,
            "estimated_tokens":   tokens,
            "fits_safely":        chars <= SAFE_LIMIT_CHARS,
            "fits_at_all":        chars <= HARD_MAX_CHARS,
            "needs_compression":  chars >  SAFE_LIMIT_CHARS,
            "data_fields":        n_fields,
            "sparse":             n_fields == 0,
        })
    except Exception as e:
        print(f"  ERROR on {pid[:8]}: {e}")

    if (i + 1) % 50 == 0:
        print(f"  ...{i+1}/{len(ALL_PATIENT_IDS)} done")

sizes_df = pd.DataFrame(raw_sizes)
valid    = sizes_df[sizes_df["raw_chars"] > 0]


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — SIZE REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("SIZE REPORT")
print("=" * 65)

n_safe    = valid["fits_safely"].sum()
n_over    = (~valid["fits_safely"] & valid["fits_at_all"]).sum()
n_wontfit = (~valid["fits_at_all"]).sum()
n_sparse  = valid["sparse"].sum()

print(f"\n  Total patients        : {len(valid)}")
print(f"\n  Raw size (chars):")
print(f"    Min    : {valid['raw_chars'].min():>10,}")
print(f"    Median : {valid['raw_chars'].median():>10,.0f}")
print(f"    Mean   : {valid['raw_chars'].mean():>10,.0f}")
print(f"    Max    : {valid['raw_chars'].max():>10,}")

print(f"\n  Context window fit (RAW, no compression):")
print(f"    Fits safely (<= {SAFE_LIMIT_CHARS:,} chars)  : {n_safe:>3} / {len(valid)}")
print(f"    Over safe limit, still fits        : {n_over:>3} / {len(valid)}")
print(f"    Won't fit at all (> {HARD_MAX_CHARS:,} chars) : {n_wontfit:>3} / {len(valid)}")

print(f"\n  Data sparsity:")
for n, label in [(3, "All 3 fields (conditions + obs + meds)"),
                 (2, "2 fields present"),
                 (1, "1 field only"),
                 (0, "All empty — sparse record")]:
    count = (valid["data_fields"] == n).sum()
    bar   = "█" * min(count // 2, 30)
    print(f"    {label:<42}: {count:>3}  {bar}")

print(f"\n  Size distribution:")
bins = [0, 5_000, 10_000, 20_000, 40_000, 80_000, 160_000, 500_000]
for lo, hi in zip(bins, bins[1:]):
    count = len(valid[(valid["raw_chars"] >= lo) & (valid["raw_chars"] < hi)])
    if count > 0:
        bar = "█" * min(count, 40)
        print(f"    {lo:>7,}–{hi:>7,} chars : {count:>3}  {bar}")

# Size distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(valid["raw_chars"], bins=30, color="#4472C4", edgecolor="white", alpha=0.85)
ax.axvline(SAFE_LIMIT_CHARS, color="#D62728", linewidth=2, linestyle="--",
           label=f"Safe limit ({SAFE_LIMIT_CHARS:,} chars)")
ax.axvline(HARD_MAX_CHARS,   color="#FF7F0E", linewidth=2, linestyle=":",
           label=f"Hard max ({HARD_MAX_CHARS:,} chars)")
ax.set_xlabel("Raw record size (chars)", fontsize=11)
ax.set_ylabel("Number of patients", fontsize=11)
ax.set_title("T5 Cohort — Raw Record Size Distribution (N=200)", fontsize=12, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# STEP 4 — VERDICT + DECIDE COMPRESSION STRATEGY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("VERDICT")
print("=" * 65)

NEEDS_COMPRESSION = n_over > 0 or n_wontfit > 0

if not NEEDS_COMPRESSION:
    print(f"\n  ALL {len(valid)} patients fit within the safe context limit.")
    print(f"  COMPRESSION NOT NEEDED.")
    print(f"\n  Strategy: pass raw records directly to LLM agents.")
    print(f"  compressed_records_cache = raw_records (all uncompressed)")
    compressed_records_cache = raw_records
    print(f"\n  GO — ready for Cell B")

else:
    print(f"\n  {n_over + n_wontfit} patients exceed the safe limit.")
    print(f"  COMPRESSION REQUIRED for those patients.")
    print(f"  Strategy: raw for patients that fit, compressed for the rest.")
    print(f"\n  --> Proceeding to STEP 5: upload MASTER_COMPRESSION_MAP")

    # ── STEP 5: Upload compression map ────────────────────────
    print("\n" + "=" * 65)
    print("STEP 5 — Upload MASTER_COMPRESSION_MAP")
    print("=" * 65)
    print("\n  Upload master_compression_map.json (or .txt)")
    uploaded_map = colab_files.upload()
    map_filename = list(uploaded_map.keys())[0]

    with open(map_filename, "r") as f:
        raw_content = f.read().strip()

    try:
        parsed = json.loads(raw_content)
    except json.JSONDecodeError:
        json_match = re.search(r'\{.*\}', raw_content, re.DOTALL)
        parsed = json.loads(json_match.group()) if json_match else {}

    # Unwrap if nested
    if "master_compression_map" in parsed:
        MASTER_COMPRESSION_MAP = parsed["master_compression_map"]
    elif "observations" in parsed and "medications" in parsed:
        MASTER_COMPRESSION_MAP = parsed
    else:
        first_val = list(parsed.values())[0]
        MASTER_COMPRESSION_MAP = first_val if isinstance(first_val, dict) else parsed

    total_descs = sum(len(v) for v in MASTER_COMPRESSION_MAP.values())
    print(f"  Loaded: {total_descs} descriptions across {len(MASTER_COMPRESSION_MAP)} tables")
    for table, descs in MASTER_COMPRESSION_MAP.items():
        print(f"    {table:<15}: {len(descs)} strings")

    # ── STEP 6: Build compressed version ──────────────────────
    print("\n" + "=" * 65)
    print("STEP 6 — Building smart_compress_patient() for T5")
    print("=" * 65)

    def get_desc_col(df):
        for col in ["DESCRIPTION", "REASONDESCRIPTION"]:
            if col in df.columns:
                return col
        raise KeyError(f"No description column found. Columns: {list(df.columns)}")

    def build_compressed_record(patient_id):
        """
        Apply framework filter + obs history cap.
        Used only when the raw record exceeds SAFE_LIMIT_CHARS.
        Data is already window-sliced — no date filtering needed.
        """
        record = (
            f"=== PATIENT RECORD: {patient_id} ===\n"
            f"=== INDEX DATE: {INDEX_DATE_LABEL.upper()} "
            f"(5 years after first T2D treatment) ===\n\n"
        )

        # Demographics — always pass through
        if "Id" in patients.columns:
            demo = patients[patients["Id"] == patient_id]
        else:
            demo = patients[patients["PATIENT"] == patient_id]
        if not demo.empty:
            keep = [c for c in demo.columns if c not in PII_COLS]
            record += f"--- DEMOGRAPHICS ---\n{demo[keep].to_string(index=False)}\n\n"
        else:
            record += "--- DEMOGRAPHICS ---\nNo demographics found.\n\n"

        meta = patient_list[patient_list["PATIENT"] == patient_id]
        if not meta.empty:
            record += f"--- INDEX DATE METADATA ---\n{meta.to_string(index=False)}\n\n"

        # Conditions — framework filter
        cond = conditions[conditions["PATIENT"] == patient_id] if not conditions.empty else pd.DataFrame()
        if not cond.empty:
            desc_col   = get_desc_col(cond)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("conditions", []))
            relevant   = cond[cond[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER"]
            keep_cols  = [c for c in cond.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else cond
            note       = (f"{len(cond)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(cond)} total, 0 matched framework — showing all")
            record += f"--- CONDITIONS [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- CONDITIONS ---\nNo records in covariate window.\n\n"

        # Observations — framework filter + last N per type
        obs = observations[observations["PATIENT"] == patient_id] if not observations.empty else pd.DataFrame()
        if not obs.empty:
            desc_col   = get_desc_col(obs)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("observations", []))
            relevant   = obs[obs[desc_col].isin(keep_descs)].copy()
            drop_cols  = ["PATIENT", "ENCOUNTER", "TYPE"]
            keep_cols  = [c for c in obs.columns if c not in drop_cols]
            if not relevant.empty:
                relevant["DATE"] = pd.to_datetime(relevant["DATE"], errors="coerce")
                relevant = relevant.sort_values("DATE", ascending=False)
                compressed = relevant.groupby(desc_col).head(MAX_OBS_HISTORY)
                compressed = compressed.sort_values([desc_col, "DATE"], ascending=[True, False])
                note = (f"{len(obs)} total → {len(relevant)} framework-relevant "
                        f"→ {len(compressed)} after last {MAX_OBS_HISTORY} per type")
                source = compressed
            else:
                note   = f"{len(obs)} total, 0 matched framework — showing all"
                source = obs
            src_cols = [c for c in keep_cols if c in source.columns]
            record += f"--- OBSERVATIONS [{note}] ---\n{source[src_cols].to_string(index=False)}\n\n"
        else:
            record += "--- OBSERVATIONS ---\nNo records in covariate window.\n\n"

        # Medications — framework filter
        med = medications[medications["PATIENT"] == patient_id] if not medications.empty else pd.DataFrame()
        if not med.empty:
            desc_col   = get_desc_col(med)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("medications", []))
            relevant   = med[med[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER", "PAYER"]
            keep_cols  = [c for c in med.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else med
            note       = (f"{len(med)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(med)} total, 0 matched — showing all")
            record += f"--- MEDICATIONS [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- MEDICATIONS ---\nNo records in covariate window.\n\n"

        # Procedures — framework filter
        proc = procedures[procedures["PATIENT"] == patient_id] if not procedures.empty else pd.DataFrame()
        if not proc.empty:
            desc_col   = get_desc_col(proc)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("procedures", []))
            relevant   = proc[proc[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER"]
            keep_cols  = [c for c in proc.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else proc
            note       = (f"{len(proc)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(proc)} total, 0 matched — showing all")
            record += f"--- PROCEDURES [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- PROCEDURES ---\nNo records in covariate window.\n\n"

        # Careplans — framework filter
        cp = careplans[careplans["PATIENT"] == patient_id] if not careplans.empty else pd.DataFrame()
        if not cp.empty:
            desc_col   = get_desc_col(cp)
            keep_descs = set(MASTER_COMPRESSION_MAP.get("careplans", []))
            relevant   = cp[cp[desc_col].isin(keep_descs)]
            drop_cols  = ["PATIENT", "ENCOUNTER"]
            keep_cols  = [c for c in cp.columns if c not in drop_cols]
            source     = relevant if not relevant.empty else cp
            note       = (f"{len(cp)} total, {len(relevant)} framework-relevant"
                          if not relevant.empty
                          else f"{len(cp)} total, 0 matched — showing all")
            record += f"--- CAREPLANS [{note}] ---\n{source[keep_cols].to_string(index=False)}\n\n"
        else:
            record += "--- CAREPLANS ---\nNo records in covariate window.\n\n"

        # Allergies + immunizations — always pass through (small)
        for section_name, df, drop_cols in [
            ("ALLERGIES",     allergies,     ["PATIENT", "ENCOUNTER"]),
            ("IMMUNIZATIONS", immunizations, ["PATIENT", "ENCOUNTER"]),
            ("ENCOUNTERS",    encounters,    ["PATIENT"]),
        ]:
            if df.empty:
                record += f"--- {section_name} ---\nSheet not present.\n\n"
                continue
            pat_df = df[df["PATIENT"] == patient_id]
            if not pat_df.empty:
                keep = [c for c in pat_df.columns if c not in drop_cols]
                record += f"--- {section_name} ({len(pat_df)} rows) ---\n{pat_df[keep].to_string(index=False)}\n\n"
            else:
                record += f"--- {section_name} ---\nNo records in covariate window.\n\n"

        return record

    def smart_compress_patient(patient_id):
        """Use raw record if it fits; apply compression only if it doesn't."""
        raw = raw_records.get(patient_id) or build_raw_record(patient_id)
        if len(raw) <= SAFE_LIMIT_CHARS:
            return raw, "raw"
        compressed = build_compressed_record(patient_id)
        return compressed, "compressed"

    print("  smart_compress_patient() built\n")

    # ── STEP 7: Build compressed_records_cache ────────────────
    print("STEP 7 — Building compressed_records_cache for all 200 patients")

    compressed_records_cache = {}
    mode_counts = {"raw": 0, "compressed": 0}

    oversized_ids = set(
        valid[valid["needs_compression"]]["patient_id"].tolist()
    )

    for i, pid in enumerate(ALL_PATIENT_IDS):
        if pid in oversized_ids:
            record, mode = smart_compress_patient(pid)
        else:
            record, mode = raw_records[pid], "raw"
        compressed_records_cache[pid] = record
        mode_counts[mode] += 1
        if (i + 1) % 50 == 0:
            print(f"  ...{i+1}/{len(ALL_PATIENT_IDS)} done")

    print(f"\n  Cache built:")
    print(f"    Raw        : {mode_counts['raw']:>3} patients")
    print(f"    Compressed : {mode_counts['compressed']:>3} patients")

    # Re-measure after compression
    final_sizes = {pid: len(rec) for pid, rec in compressed_records_cache.items()}
    still_over  = [pid for pid, sz in final_sizes.items() if sz > HARD_MAX_CHARS]
    if still_over:
        print(f"\n  WARNING: {len(still_over)} patients STILL over hard limit after compression.")
        print(f"  Consider reducing MAX_OBS_HISTORY from {MAX_OBS_HISTORY} to 3 and re-running.")
        for pid in still_over:
            print(f"    {pid[:8]}  {final_sizes[pid]:,} chars")
    else:
        print(f"\n  All patients within hard limit after compression.")

    print(f"\n  GO — ready for Cell B")


# ─────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("SUMMARY")
print("=" * 65)
print(f"  compressed_records_cache : {len(compressed_records_cache)} patients")
print(f"  Index date label         : {INDEX_DATE_LABEL}")
print(f"  Compression used         : {NEEDS_COMPRESSION}")
print(f"\n  Do NOT restart the runtime before running Cell B.")
print(f"  compressed_records_cache must stay in memory.")

In [ ]:
# ── Upload Frozen Framework ───────────────────────────────────
from google.colab import files as colab_files

print("📤 Upload your consolidated_framework_v2.txt file...")
uploaded = colab_files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r") as f:
    CONSOLIDATED_FRAMEWORK = f.read()

print(f"✅ CONSOLIDATED_FRAMEWORK loaded ({len(CONSOLIDATED_FRAMEWORK):,} chars)")
print(f"   Preview: {CONSOLIDATED_FRAMEWORK[:200]}...")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  RELOAD CELL — Restore open-weight results from zip         ║
# ║  Run this after runtime disconnect                          ║
# ╚══════════════════════════════════════════════════════════════╝
import zipfile, io, os, glob, re, json
import pandas as pd
from google.colab import files as colab_files

# ── STEP 1: Upload checkpoint zip ─────────────────────────────
print("📤 Upload your open-weight checkpoint zip...")
uploaded = colab_files.upload()

if not uploaded:
    print("❌ No file uploaded.")
else:
    zip_name  = list(uploaded.keys())[0]
    zip_bytes = list(uploaded.values())[0]
    print(f"   Uploaded: {zip_name} ({len(zip_bytes)/1024/1024:.1f} MB)")

    # ── STEP 2: Extract ────────────────────────────────────────
    EXTRACT_DIR = "/content/pipeline_outputs"
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"   Extracted to {EXTRACT_DIR}")

    # ── STEP 3: Find the consolidated folder ──────────────────
    INDEX_DATE_LABEL = "T5_reclassification_local"
    INDEX_SUBDIR     = f"{EXTRACT_DIR}/phase2/{INDEX_DATE_LABEL}"

    # Try to find it even if folder structure is slightly different
    if not os.path.isdir(INDEX_SUBDIR):
        for root, dirs, files in os.walk(EXTRACT_DIR):
            if INDEX_DATE_LABEL in root:
                INDEX_SUBDIR = root
                break

    print(f"   INDEX_SUBDIR: {INDEX_SUBDIR}")

    OW_CONSOLIDATED_DIR = f"{INDEX_SUBDIR}/consolidated"
    ow_files = glob.glob(f"{OW_CONSOLIDATED_DIR}/classification_*.txt")
    print(f"   Consolidated files found: {len(ow_files)}")

    # ── STEP 4: Restore completed_patients set ─────────────────
    # Try checkpoint.json first
    checkpoint_path = f"{INDEX_SUBDIR}/checkpoint.json"
    completed_patients = set()

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            ck = json.load(f)
        completed_patients = set(ck.get("completed_patients", []))
        print(f"   Restored from checkpoint.json: {len(completed_patients)} patients")
    else:
        # Fall back to inferring from consolidated files
        for fp in ow_files:
            m = re.search(r'classification_([a-f0-9\-]+)', os.path.basename(fp))
            if m:
                short_id = m.group(1)
                # Try to match back to full UUID via MASTER_GT if available
                completed_patients.add(short_id)
        print(f"   checkpoint.json not found — inferred {len(completed_patients)} from files")

    # ── STEP 5: Parse all OW classifications back into memory ──
    def normalise_tier(raw):
        if pd.isna(raw) or raw is None: return None
        v = str(raw).lower().strip()
        if "advanced" in v or "critical" in v: return "Advanced/Critical"
        if "moderate" in v:                    return "Moderate Complications"
        if "mild" in v:                        return "Mild Complications"
        if "baseline" in v:                    return "Baseline T2D"
        return None

    def extract_tier_from_txt(filepath):
        try:
            with open(filepath, "r") as f:
                text = f.read()
        except:
            return None
        if "===FINAL===" in text:
            block = text.split("===FINAL===")[-1]
            for line in block.split("\n"):
                m = re.match(r'^FOUR_TIER\s*:\s*(.+)$',
                             line.replace("**","").strip(), re.IGNORECASE)
                if m:
                    return normalise_tier(m.group(1).strip())
        return normalise_tier(text[-500:])

    ow_results = {}
    for fp in ow_files:
        m = re.search(r'classification_([a-f0-9\-]+)', os.path.basename(fp))
        if m:
            ow_results[m.group(1)] = extract_tier_from_txt(fp)

    # ── STEP 6: Reload ALL_PATIENT_IDS if MASTER_GT available ──
    if "MASTER_GT" in dir():
        MASTER_GT["OW_TIER"] = MASTER_GT["PATIENT"].str[:8].map(ow_results)

        # Restore completed_patients as full UUIDs if inferred from short IDs
        if all(len(p) == 8 for p in completed_patients):
            short_to_full = dict(zip(
                MASTER_GT["PATIENT"].str[:8],
                MASTER_GT["PATIENT"]
            ))
            completed_patients = set(
                short_to_full[s] for s in completed_patients
                if s in short_to_full
            )
            print(f"   Converted short IDs → full UUIDs: {len(completed_patients)}")

        ALL_PATIENT_IDS = MASTER_GT["PATIENT"].tolist()
        print(f"   ALL_PATIENT_IDS restored: {len(ALL_PATIENT_IDS)} patients")
        print(f"   OW_TIER mapped to MASTER_GT: {MASTER_GT['OW_TIER'].notna().sum()} patients")
    else:
        print("   ⚠️  MASTER_GT not in memory — upload it separately to enable full restore")
        print("      ALL_PATIENT_IDS not restored yet")

    # ── STEP 7: Summary ────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  RESTORE SUMMARY")
    print(f"{'='*55}")
    print(f"  INDEX_SUBDIR        : {INDEX_SUBDIR}")
    print(f"  OW files on disk    : {len(ow_files)}")
    print(f"  OW results parsed   : {len(ow_results)}")
    print(f"  completed_patients  : {len(completed_patients)}")
    if "MASTER_GT" in dir():
        print(f"  OW_TIER in MASTER   : {MASTER_GT['OW_TIER'].notna().sum()}")
        print(f"  Remaining patients  : {len(ALL_PATIENT_IDS) - len(completed_patients)}")
    print(f"\n  ✅ Ready to continue — run the identify cell next")
    print(f"{'='*55}")

In [ ]:
# ============================================================
# CELL 7: Phase 2 Pipeline — Open Weight Committee (Optimised)
# ============================================================

from crewai import Agent, Task, Crew
import time, json, zipfile, math, re, os
from google.colab import files as colab_files

# ── CONFIG ────────────────────────────────────────────────────
INDEX_DATE_LABEL   = "T5_reclassification_local"
INDEX_DATE_DISPLAY = "T5 — 5-Year Reclassification (Local Open-Weight)"

BATCH_SIZE       = 421
CHECKPOINT_EVERY = 20

OUTPUT_DIR   = "/content/pipeline_outputs"
INDEX_SUBDIR = f"{OUTPUT_DIR}/phase2/{INDEX_DATE_LABEL}"
os.makedirs(INDEX_SUBDIR, exist_ok=True)

VALID_TIERS = [
    "Baseline T2D",
    "Mild Complications",
    "Moderate Complications",
    "Advanced/Critical",
]
COMPLEX_TIERS = {"Moderate Complications", "Advanced/Critical"}

# ── Utilities ─────────────────────────────────────────────────
def save_and_track(subpath, filename, content):
    path = f"{INDEX_SUBDIR}/{subpath}/{filename}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if isinstance(content, (dict, list)):
        with open(path + ".json", "w") as f:
            json.dump(content, f, indent=2)
    with open(path + ".txt", "w") as f:
        f.write(json.dumps(content, indent=2) if isinstance(content, (dict, list)) else str(content))

def softmax_scores(s_baseline, s_mild, s_moderate, s_advanced):
    scores = [s_baseline or 0, s_mild or 0, s_moderate or 0, s_advanced or 0]
    exps   = [math.exp(s) for s in scores]
    total  = sum(exps)
    probs  = [round(e / total, 4) for e in exps]
    return {
        "prob_baseline_t2d"          : probs[0],
        "prob_mild_complications"    : probs[1],
        "prob_moderate_complications": probs[2],
        "prob_advanced_critical"     : probs[3],
    }

def parse_score(parsed_dict, key):
    try:
        return int(float(parsed_dict.get(key, "").strip()))
    except (ValueError, TypeError, AttributeError):
        return None

def download_checkpoint_zip(completed_set, all_results, errors):
    checkpoint_meta = {
        "index_date"         : INDEX_DATE_LABEL,
        "completed_patients" : list(completed_set),
        "total_patients"     : len(ALL_PATIENT_IDS),
        "errors"             : len(errors),
        "timestamp"          : time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(f"{INDEX_SUBDIR}/checkpoint.json", "w") as f:
        json.dump(checkpoint_meta, f, indent=2)
    save_and_track("metadata", "pipeline_results_partial", all_results)
    if errors:
        save_and_track("metadata", "pipeline_errors_partial", errors)

    zip_name = f"/content/pipeline_checkpoint_{INDEX_DATE_LABEL}.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, files_list in os.walk(INDEX_SUBDIR):
            for fname in files_list:
                fpath   = os.path.join(root, fname)
                arcname = os.path.relpath(fpath, OUTPUT_DIR)
                zf.write(fpath, arcname)
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"  Checkpoint saved: {len(completed_set)} patients, {size_mb:.1f} MB")
    colab_files.download(zip_name)

# ── Resume check ──────────────────────────────────────────────
if "completed_patients" not in dir() or not isinstance(completed_patients, set):
    completed_patients = set()

remaining_all = [pid for pid in ALL_PATIENT_IDS if pid not in completed_patients]
remaining     = remaining_all[:BATCH_SIZE]

print(f"{'=' * 65}")
print(f"PHASE 2 — {INDEX_DATE_DISPLAY}")
print(f"{'=' * 65}")
print(f"  Total patients  : {len(ALL_PATIENT_IDS)}")
print(f"  Already done    : {len(completed_patients)}")
print(f"  Remaining total : {len(remaining_all)}")
print(f"  This batch      : {len(remaining)}")
print(f"{'=' * 65}")

if not remaining:
    print("All patients already completed!")
else:
    # ══════════════════════════════════════════════════════════
    # AGENT DEFINITIONS — defined ONCE, reused for every patient
    # ══════════════════════════════════════════════════════════

    assessor_gemma = Agent(
        role="Dr. A — Clinical Informatician",
        goal="Classify T2D severity at T5 using the framework. Be methodical and evidence-based.",
        backstory="Clinical informatician specialising in EHR-based T2D phenotyping. Conservative — only count what is explicitly documented.",
        verbose=False,       # ← changed to False: verbose=True floods the log and slows Colab
        allow_delegation=False,
        llm=llm_gemma
    )

    assessor_qwen = Agent(
        role="Dr. B — Consultant Endocrinologist",
        goal="Classify T2D severity at T5 using the framework. Take a holistic, outcomes-focused approach.",
        backstory="Consultant endocrinologist with 20 years T2D experience. Considers the full 5-year clinical trajectory.",
        verbose=False,
        allow_delegation=False,
        llm=llm_qwen
    )

    consolidator = Agent(
        role="Chief Medical Informatician — Final Arbiter",
        goal="Review both assessors' classifications, resolve disagreements, produce final severity classification.",
        backstory="Chief medical informatician responsible for final phenotyping decisions.",
        verbose=False,
        allow_delegation=False,
        llm=llm_llama
    )

    # ══════════════════════════════════════════════════════════
    # MAIN LOOP
    # ══════════════════════════════════════════════════════════
    all_results               = []
    errors                    = []
    patients_since_checkpoint = 0
    pipeline_start            = time.time()

    for idx, patient_id in enumerate(remaining):
        patient_short = patient_id[:8]
        raw_record    = compressed_records_cache[patient_id]
        overall_idx   = len(completed_patients) + 1

        print(f"\n{'=' * 65}")
        print(f"Patient {overall_idx}/{len(ALL_PATIENT_IDS)}: {patient_short} "
              f"({len(raw_record):,} chars, ~{len(raw_record)//4:,} tokens)")
        print(f"{'=' * 65}")

        # ── ASSESSOR PROMPT (shared by both Dr. A and Dr. B) ──
        ASSESS_PROMPT = f"""Classify this T2D patient's severity at their T5 (5-year) reclassification point.

TIERS: Tier 1=Baseline T2D | Tier 2=Mild Complications | Tier 3=Moderate Complications | Tier 4=Advanced/Critical
Rule: Classify based on ALL available evidence — medications, procedures, conditions, and lab values — not just explicit complication codes. Only default to Baseline T2D if the record is truly empty of any complication signals. Never respond with "Insufficient Data".

=== FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}

=== PATIENT RECORD ===
{raw_record}

Respond with:
1. Domain assessment (1-2 sentences per domain only). After your domain assessment, explicitly state:
"Concurrent Tier 3 domains identified: [list them]"
and "Rule 0a check: [met / not met]".
This must appear before your final tier.
2. Any notable clinical observations not covered by the framework
3. Final tier + binary classification (COMPLEX or NOT_COMPLEX) + confidence (High/Medium/Low)
4. Scores (0-100 each, must sum to approximately 100):
SCORE_BASELINE_T2D: [0-100]
SCORE_MILD_COMPLICATIONS: [0-100]
SCORE_MODERATE_COMPLICATIONS: [0-100]
SCORE_ADVANCED_CRITICAL: [0-100]
"""

        # ── CONSOLIDATOR PROMPT ────────────────────────────────
        CONSOLIDATE_PROMPT = f"""Review the two assessors' outputs for patient {patient_id} and produce the final classification.
Before finalising, perform a structured self-check grounded in the framework's own decision logic:

1. Re-read the KEY_EVIDENCE from each assessor. Count how many distinct domain criteria
   at Tier 3 level or above are mentioned across both assessments combined.

2. Apply Framework Rule 0a: if two or more concurrent Tier 3 domain criteria are present
   in the record, the framework mandates upward reclassification to Tier 4. Have both
   assessors accounted for this rule, or did either stop at Tier 3 without checking it?

3. Apply Framework Rule 0: the final tier is the highest tier triggered by any single domain —
   not an average. If one assessor found a Tier 4 signal and the other found Tier 3,
   the correct resolution is Tier 4 unless the Tier 4 signal is explicitly contradicted
   by the record.

4. Check for under-classification risk: if the record contains insulin dependence alongside
   any documented complication, confirm this has been correctly weighted per the
   pharmacotherapy proxy in Domain 7.

5. Check for over-classification risk: confirm that a Tier 4 assignment is not based
   on a single isolated finding without corroborating domain evidence.

Output the ===FINAL=== block first, then write 2-3 sentences summarising your self-check

===FINAL===
PATIENT_ID: {patient_id}
FOUR_TIER: [Baseline T2D / Mild Complications / Moderate Complications / Advanced/Critical]
BINARY: [COMPLEX / NOT_COMPLEX]
ASSESSOR_A_TIER: [Dr. A tier]
ASSESSOR_B_TIER: [Dr. B tier]
CONFIDENCE: [High / Medium / Low]
AGREEMENT: [Full / Partial / None]
INDEX_DATE_CONTEXT: {INDEX_DATE_LABEL}
KEY_EVIDENCE: [One decisive sentence]
SCORE_BASELINE_T2D: [0-100]
SCORE_MILD_COMPLICATIONS: [0-100]
SCORE_MODERATE_COMPLICATIONS: [0-100]
SCORE_ADVANCED_CRITICAL: [0-100]
===END===
"""

        task_assess_gemma = Task(
            description=ASSESS_PROMPT,
            expected_output="Brief domain assessment, final tier, binary, confidence, and four scores.",
            agent=assessor_gemma,
            async_execution=False
        )

        task_assess_qwen = Task(
            description=ASSESS_PROMPT,
            expected_output="Brief domain assessment, final tier, binary, confidence, and four scores.",
            agent=assessor_qwen,
            async_execution=False
        )

        task_consolidate = Task(
            description=CONSOLIDATE_PROMPT,
            expected_output="===FINAL=== block followed by 1-2 sentences of resolution reasoning.",
            agent=consolidator,
            context=[task_assess_gemma, task_assess_qwen]
        )

        crew = Crew(
            agents=[assessor_gemma, assessor_qwen, consolidator],
            tasks=[task_assess_gemma, task_assess_qwen, task_consolidate],
            verbose=False    # ← keeps console clean; individual task outputs still saved to disk
        )

        patient_start = time.time()

        try:
            crew_result     = crew.kickoff()
            patient_elapsed = time.time() - patient_start

            save_and_track("gemma",  f"assessment_{patient_short}", str(task_assess_gemma.output))
            save_and_track("qwen",   f"assessment_{patient_short}", str(task_assess_qwen.output))

            completed_patients.add(patient_id)
            patients_since_checkpoint += 1

            if hasattr(crew_result, 'raw') and crew_result.raw:
                result_text = crew_result.raw
            elif hasattr(crew_result, 'result') and crew_result.result:
                result_text = crew_result.result
            else:
                result_text = str(crew_result)

            save_and_track("consolidated", f"classification_{patient_short}", result_text)

            def parse_final_block(text):
                parts = text.split("===FINAL===")
                if len(parts) < 2:
                    return {}, "NO_FINAL_BLOCK"
                block = parts[-1].split("===END===")[0] if "===END===" in parts[-1] else "\n".join(parts[-1].strip().split("\n")[:30])
                result = {}
                for line in block.strip().split("\n"):
                    line = line.replace("**", "").strip()
                    m = re.match(r'^([a-zA-Z0-9_]+)\s*:\s*(.*)$', line)
                    if m:
                        key = m.group(1).strip().upper()
                        val = re.sub(r'^[\[\(\`\"\'](.*?)[\]\)\`\"\']$', r'\1', m.group(2).strip()).strip()
                        result[key] = val
                return result, "OK"

            parsed, parse_status = parse_final_block(result_text)

            def extract_tier(raw_val):
                if not raw_val: return "PARSE_ERROR"
                v = str(raw_val).lower()
                if "advanced" in v or "critical" in v: return "Advanced/Critical"
                if "moderate" in v:                    return "Moderate Complications"
                if "mild" in v:                        return "Mild Complications"
                if "baseline" in v:                    return "Baseline T2D"
                return "PARSE_ERROR"

            four_tier = extract_tier(parsed.get("FOUR_TIER", ""))

            if four_tier == "PARSE_ERROR":
                m = re.search(r'(?:FOUR_TIER|Final Tier|Classification)\s*[:\|-]\s*\**([A-Za-z/ ]+)', result_text, re.IGNORECASE)
                if m:
                    four_tier = extract_tier(m.group(1))
                if four_tier == "PARSE_ERROR":
                    four_tier = extract_tier(result_text[-1000:])

            binary = parsed.get("BINARY", "")
            if "not_complex" in binary.lower() or "not complex" in binary.lower():
                binary = "NOT_COMPLEX"
            elif "complex" in binary.lower():
                binary = "COMPLEX"
            else:
                binary = "PARSE_ERROR"

            s_baseline  = parse_score(parsed, "SCORE_BASELINE_T2D")
            s_mild      = parse_score(parsed, "SCORE_MILD_COMPLICATIONS")
            s_moderate  = parse_score(parsed, "SCORE_MODERATE_COMPLICATIONS")
            s_advanced  = parse_score(parsed, "SCORE_ADVANCED_CRITICAL")
            softmax     = softmax_scores(s_baseline, s_mild, s_moderate, s_advanced)

            result_record = {
                "patient_id"          : patient_id,
                "index_date_type"     : INDEX_DATE_LABEL,
                "four_tier"           : four_tier,
                "binary"              : binary,
                "assessor_a_tier"     : extract_tier(parsed.get("ASSESSOR_A_TIER", "")),
                "assessor_b_tier"     : extract_tier(parsed.get("ASSESSOR_B_TIER", "")),
                "confidence"          : parsed.get("CONFIDENCE", ""),
                "agreement"           : parsed.get("AGREEMENT", ""),
                "key_evidence"        : parsed.get("KEY_EVIDENCE", ""),
                "score_baseline_t2d"  : s_baseline,
                "score_mild"          : s_mild,
                "score_moderate"      : s_moderate,
                "score_advanced"      : s_advanced,
                **softmax,
                "parse_status"        : parse_status,
                "elapsed_seconds"     : round(patient_elapsed, 1),
                "record_chars"        : len(raw_record),
            }

            all_results.append(result_record)
            save_and_track("metadata", f"result_{patient_short}", result_record)

            # ── Console summary ───────────────────────────────
            agreement_flag = "✅" if parsed.get("AGREEMENT", "").lower() == "full" else "⚠️"
            print(f"  ✅ Done in {patient_elapsed:.0f}s")
            print(f"  Tier: {four_tier} | Binary: {binary} | Confidence: {parsed.get('CONFIDENCE','?')}")
            print(f"  {agreement_flag} Agreement: {parsed.get('AGREEMENT','?')} | Key: {parsed.get('KEY_EVIDENCE','')[:80]}")
            print(f"  Scores → Baseline:{s_baseline} Mild:{s_mild} Moderate:{s_moderate} Advanced:{s_advanced}")

        except Exception as e:
            patient_elapsed = time.time() - patient_start
            print(f"  ❌ ERROR for {patient_short}: {e}")
            errors.append({"patient_id": patient_id, "error": str(e), "elapsed": round(patient_elapsed, 1)})

        # ── Checkpoint ────────────────────────────────────────
        if patients_since_checkpoint >= CHECKPOINT_EVERY:
            print(f"\n  💾 Saving checkpoint ({len(completed_patients)} done)...")
            download_checkpoint_zip(completed_patients, all_results, errors)
            patients_since_checkpoint = 0

    # ── Final save ────────────────────────────────────────────
    total_elapsed = time.time() - pipeline_start
    print(f"\n{'=' * 65}")
    print(f"BATCH COMPLETE")
    print(f"  Processed : {len(remaining)} patients")
    print(f"  Errors    : {len(errors)}")
    print(f"  Total time: {total_elapsed/60:.1f} min ({total_elapsed/len(remaining):.0f}s/patient avg)")
    print(f"{'=' * 65}")

    save_and_track("metadata", "pipeline_results_final", all_results)
    if errors:
        save_and_track("metadata", "pipeline_errors_final", errors)
    download_checkpoint_zip(completed_patients, all_results, errors)